# 6주차 4차시: 하이브리드 검색 · 평가 · 추천 시스템

| 주제 | 내용 |
|---|---|
| Smart Filter | LLM 으로 자연어 쿼리 → JSON 필터 추출 |
| Hybrid Search | 벡터 + BM25 가중 결합 (`alpha`) |
| 평가 지표 | Hit Rate@K · MRR@K |
| Reranking | MMR · LLM Listwise |
| CBF | 아이템 속성 벡터 + 코사인 유사도 추천 |


In [1]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install finance-datareader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 1.2 MB/s eta 0:00:00


In [32]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_community.vectorstores import FAISS as LangchainFAISS
from langchain_core.documents import Document

# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

In [17]:
import FinanceDataReader as fdr

## (전일 복습) 데이터 준비

4차시 실습에서 쓸 `documents` · `vectorstore` · `bm25` 를 전일 3차시 흐름대로 재구축하는 구간.

- ETF 목록 수집 → `compute_metrics` · `risk_from_vol` 으로 지표 계산
- LLM `generate_description` 으로 자연어 설명 생성 → `Document` 구성
- `FAISS.from_documents` 로 벡터 인덱스, Kiwi 토크나이저로 BM25 corpus 구축


In [33]:
etf_listing = fdr.StockListing('ETF/KR')

name_by_ticker = dict(zip(etf_listing['Symbol'].astype(str).str.zfill(6), etf_listing['Name']))


def risk_from_vol(vol):
  if vol < 5 : return "낮음"
  if vol < 15 : return "약간 낮음"
  if vol < 25 : return "중간"
  return "높음"
def compute_metrics(df):
    close = df['Close'].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 -1)*100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100

    return {'return_1y': round(ann_ret, 2), 'volatility': round(ann_vol, 2), 'mdd' : round(mdd, 2)}

# 티커(번호)를 입력하여, fdr.DataReader()를 이용해 주가 정보의 df를 반환하는 함수 작성
def collect_etf_price(ticker, days=365):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')

    df = fdr.DataReader(ticker, start_str, end_str)

    return {'status' : 'real', 'data': df, 'ticker' : ticker}

collect_etf_price('069500')

tickers_info = {
    "069500": {"category": "국내주식", "expense_ratio": 0.15, "dividend_yield": 1.8,
               "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"]},
    "379800": {"category": "해외주식", "expense_ratio": 0.05, "dividend_yield": 0.0,
               "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"]},
    "411060": {"category": "배당",     "expense_ratio": 0.01, "dividend_yield": 3.5,
               "keywords": ["미국", "배당", "배당성장", "저비용", "안정"]},
    "305540": {"category": "테마",     "expense_ratio": 0.45, "dividend_yield": 0.0,
               "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"]},
    "379810": {"category": "해외주식", "expense_ratio": 0.07, "dividend_yield": 0.0,
               "keywords": ["미국", "나스닥", "기술주", "성장", "IT"]},
    "381180": {"category": "해외주식", "expense_ratio": 0.49, "dividend_yield": 0.0,
               "keywords": ["반도체", "AI", "미국", "기술주", "고위험"]},
}

etf_data =[]
for t, info in tickers_info.items():
    result = collect_etf_price(t, days=365)
    # {'status' : 'real', 'data' : df, 'ticker' : ticker}
    metrics = compute_metrics(result['data'])
    name = name_by_ticker.get(t, f"ETF_{t}")
    etf_data.append({
        "ticker" : t,
        "name" : name,
        **info,
        **metrics,
        'risk_level' : risk_from_vol(metrics['volatility']),
        'data_staus' : result['status']
    })

documents = []

# 데이터를 LLM을 활용해 생성 (description 생성)
  # 단순히 keyword를 가져오면 리스트일 테니까, 리스트를 연결해주는 '.'.join()을 사용
def generate_description(etf):

    prompt = f"""다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.
    이름 : {etf['name']}
    카테고리 : {etf['category']}
    키워드 : {','.join(etf['keywords'])}
    수수료 : {etf['expense_ratio']}% / 배당수익률: {etf['dividend_yield']}$
    1년수익률 : {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%
    """

    return llm.invoke([{'role' : 'user', 'content' : prompt}]).content.strip()

for etf in etf_data:
  desc = generate_description(etf)
  text = (f"{etf['name']} ({etf['category']}): {desc}"
        f"키워드: {'.'.join(etf['keywords'])}"
        f"수수료: {etf['expense_ratio']}%, 배당수익률:{etf['dividend_yield']}%"
        f"수익률: {etf['return_1y']}% / 변동성: {etf['volatility']}% / MDDD : {etf['mdd']}%")

  metadata = {k:v for k, v in etf.items() if k!='keywords'}
  metadata['keywords'] = '.'.join(etf['keywords'])

  documents.append(Document(page_content=text, metadata = metadata))

In [38]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [39]:
vectorstore = LangchainFAISS.from_documents(documents, embeddings)

In [5]:
# 메타데이터 필터링
def filtered_search(vectorstore, query, filters=None, k = 5, fetch_k = 20): # filters = {"expense_ratio": 0.1, "category": "해외주식"} 등이 예시임
  results = vectorstore.similarity_search_with_score(query, fetch_k) # 20개를 먼저 유사한걸 뽑아옴
  if not filters:
    return results[:k]

  filtered = []
  for doc, score in results:
    match = True
    for key, condition in filters.items():
      val = doc.metadata.get(key) # 메타데이터의 카테고리(key) 및 value

      # condition이 딕셔너리 형태일 때 (예: {"less_than": 0.5})
      if isinstance(condition, dict):
          if 'less_than' in condition:
              if not (val < condition['less_than']):
                  match = False
                  break

          if 'greater_than' in condition:
              if not (val > condition['greater_than']):
                  match = False
                  break

      # condition이 일반 값일 때 (예: "해외주식")
      else:
          if val != condition:
              match = False
              break

      if match:
        filtered.append((doc, score)) # 필터에 속하면 추가


  return filtered[:k]

## 스마트 필터 검색 (LLM Filter Extraction)

사용자의 자연어 쿼리에서 LLM 이 JSON 형식의 필터 조건을 추출 → 메타데이터 필터 검색에 그대로 주입.

- 입력: `"운용보수 0.1% 이하이면서 배당 3% 이상인 ETF"`
- 출력: `{"category": "...", "expense_ratio": {"less_than": 0.1}, "dividend_yield": {"greater_than": 3}}`
- 구조화된 필터 추출로 검색 정확도와 해석 가능성을 동시에 확보


In [13]:
def smart_filtered_search(query):
  prompt = f"""사용자 쿼리에서 ETF 검색 필터를 추출하세요.

  쿼리 : {query}

  사용 가능한 필터 필드:

  - category : 국내주식, 해외주식, 배당, 테마 ...
  - risk_level : 낮음, 중간, 높음
  - expense_ratio : 숫자 (%) (% "less_than"/"greater_than")
  - dividened_yield : 숫자 (%)
  - return_1y : 숫자 (%)
  - volatility : 숫자 (%)

  JSON 형식으로만 응답하세요. 필터가 없으면 {{}}.
  예, {{"category": "해외주식", "expense_ratio": {{"greater_than": 0.5}}}}"""

  response = llm.invoke([{'role' : 'user', 'content' : prompt}]).content
  try:
    if "```" in response:
      response = response.split("```")[1].replace("json","") # 아래처럼 ``` 으로 감싸진 것(즉 [0]과 [-1]을 제거), json을 공백("")으로 변환
    return json.loads(response)
  except Exception:
    return {}

In [ ]:
# ```json
# {
#     "category": "해외주식", "expense_ratio": {"less_than":0.5}
# }
# ```

In [14]:
test_query = "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"
filters = smart_filtered_search(test_query) # 즉 LLM이 필터를 직접 만들어준다는 것, (filtered_search에서 사용가능하도록)
filters

{'expense_ratio': {'less_than': 0.1}, 'dividened_yield': {'greater_than': 3}}

### 스마트 필터 + 문서 검색 통합 — `smart_document_search`

- 쿼리 → `smart_filtered_search` 로 필터 추출 → `filtered_search` 로 문서 검색
- 한 함수 호출로 자연어 → 구조화 → 검색 파이프라인 완성


In [23]:
def smart_document_search(query, vectorstore, k=3):
    filters = smart_filtered_search(query)

    results = filtered_search(
        vectorstore=vectorstore,
        query=query,
        filters=filters,
        k=k,
        fetch_k=max(20, k * 5)
    )

    if not results:
      print("필터 조건에 맞는 결과가 없습니다. 필터 없이 재검색...")
      results = filtered_search(vectorstore, query, k=k)

    print(f"결과 : {len(results)}개")
    for doc, score in results:
      m = doc.metadata
      print(m)

    return results

In [42]:
smart_document_search("위험 낮고 배당 2% 이상인 ETF", vectorstore)

필터 조건에 맞는 결과가 없습니다. 필터 없이 재검색...
결과 : 3개
{'ticker': '305540', 'name': 'TIGER 2차전지테마', 'category': '테마', 'expense_ratio': 0.45, 'dividend_yield': 0.0, 'return_1y': np.float64(102.47), 'volatility': np.float64(47.5), 'mdd': -23.19, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '2차전지.배터리.테마.성장.고위험'}
{'ticker': '411060', 'name': 'ACE KRX금현물', 'category': '배당', 'expense_ratio': 0.01, 'dividend_yield': 3.5, 'return_1y': np.float64(59.23), 'volatility': np.float64(32.95), 'mdd': -22.76, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '미국.배당.배당성장.저비용.안정'}
{'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(169.62), 'volatility': np.float64(31.34), 'mdd': -10.35, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '반도체.AI.미국.기술주.고위험'}


[(Document(id='2dbb73d3-ba27-423e-8522-cfe9928f7ab1', metadata={'ticker': '305540', 'name': 'TIGER 2차전지테마', 'category': '테마', 'expense_ratio': 0.45, 'dividend_yield': 0.0, 'return_1y': np.float64(102.47), 'volatility': np.float64(47.5), 'mdd': -23.19, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '2차전지.배터리.테마.성장.고위험'}, page_content='TIGER 2차전지테마 (테마): TIGER 2차전지테마 ETF는 2차전지 및 배터리 관련 기업에 투자하여 성장 잠재력을 추구하는 고위험 테마 ETF로, 최근 1년간 102.47%의 수익률을 기록하였고, 수수료는 0.45%입니다. 변동성이 47.5%로 높아 가격 변동에 유의해야 하며, 배당수익률은 없는 특징이 있습니다.키워드: 2차전지.배터리.테마.성장.고위험수수료: 0.45%, 배당수익률:0.0%수익률: 102.47% / 변동성: 47.5% / MDDD : -23.19%'),
  np.float32(0.82412434)),
 (Document(id='7c8e2539-eb83-4ce0-b6ee-89c60e1aebd1', metadata={'ticker': '411060', 'name': 'ACE KRX금현물', 'category': '배당', 'expense_ratio': 0.01, 'dividend_yield': 3.5, 'return_1y': np.float64(59.23), 'volatility': np.float64(32.95), 'mdd': -22.76, 'risk_level': '높음', 'data_staus': 'real', 'keywords': '미국.배당.배당성장.저비용.안정'}, page_content='ACE KRX금현물 (배당): ACE

## 하이브리드 검색

벡터 검색(의미) + BM25(키워드) 를 가중합으로 결합.

- `hybrid_search(query, alpha, k, filters)`
- `alpha` = 1: 벡터 전용 / `alpha` = 0: BM25 전용 / 중간값: 가중 앙상블
- 고유명사/숫자(BM25 강점) + 의미 유사(벡터 강점) 둘 다 포착


In [43]:
def bm25_search(query, k=5):
  tokens = kiwi_tokenize(query)
  scores = bm25.get_scores(tokens)
  top_idx = np.argsort(scores)[::-1][:k]

  return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]


In [46]:
def hybrid_search(query, alpha = 0.5, k=5, filters = None):
  vec_results = vectorstore.similarity_search_with_score(query, k=20)
  vec_scores = {}
  for doc, dist in vec_results:
    vec_scores[doc.metadata['name']] = 1 / (1+dist)

  # vec_scores = {'TIGER 2차전지테마' : 0.5, 'TIGER 3차전지테마' : 0.9}

  bm25_results = bm25_search(query, k=20)
  bm25_scores = {} # 1 ~ 2, 1.5 -> 0.5
  for doc, score in bm25_results:
    bm25_scores[doc.metadata['name']] = score

  def normalize(scores):
    if not scores:
      return scores

    vals = list(scores.values())
    min_, max_ = min(vals), max(vals)
    rng = max_ - min_ if max_ != min_ else 1.0
    return {k: (v-min_)/rng  for k, v in scores.items()}

  vec_norm = normalize(vec_scores)  # vec_norm = {'TIGER 2차전지테마' : 0.5, 'TIGER 3차전지테마' : 0.9}
  bm25_norm = normalize(bm25_scores) # bm25_norm = {'TIGER 2차전지테마' : 0.3, 'TIGER 3차전지테마' : 0.7}

  all_names = set(vec_norm) | set(bm25_norm)
  combined = {}
  for name in all_names:
      v = vec_norm.get(name, 0)
      b = bm25_norm.get(name, 0)
      combined[name] = alpha  * v + (1-alpha) * b  # combined = {'TIGER 2차전지테마' : 0.4, 'TIGER 3차전지테마' : ...}

  if filters:
      name_to_doc = {d.metadata['name'] : d  for d in documents}
      filtered = {}
      for name, score in combined.items():
          doc = name_to_doc.get(name)
          if not doc:
              continue

          match = True
          for key, condition in filters.items():
              val = doc.metadata.get(key)
              if isinstance(condition, dict):
                  if "less_than" in condition and val > condition["less_than"]:
                      match = False
                  if "greater_than" in condition and val < condition["greater_than"]:
                      match = False
              elif val != condition:
                  match = False
          if match:
              filtered[name] = score
      combined = filtered

  sorted_results = sorted(combined.items(), key=lambda x : x[1], reverse=True)
  return sorted_results[:k]


In [49]:
!pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 11.9 MB/s eta 0:00:00


In [50]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

kiwi = Kiwi()
def kiwi_tokenize(text):
  tokens = kiwi.tokenize(text)
  result = []
  for token in tokens:
    if token.tag in ("SL", "SN", "NNG", "NNP"):
      result.append(token.form.lower())

  return result

In [53]:
corpus = [kiwi_tokenize(doc.page_content) for doc in documents]
bm25 = BM25Okapi(corpus)

In [54]:
query = "미국 기술 성장 ETF"
for alpha in [0.0, 0.5, 1.0]:
  results = hybrid_search(query, alpha = alpha, k= 3, filters=None)
  names = [r[0] for r in results]
  print(f"alpha: {alpha} : {names}")

alpha: 0.0 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'KODEX 미국S&P500']
alpha: 0.5 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'ACE KRX금현물']
alpha: 1.0 : ['KODEX 미국나스닥100', 'TIGER 미국필라델피아반도체나스닥', 'TIGER 2차전지테마']


## 검색 성능 평가

쿼리-정답 쌍을 만들어 벡터·BM25·하이브리드 세 검색기를 정량 비교.

- `eval_queries` = `[{"query": ..., "relevant": [정답 ETF 이름, ...]}, ...]`
- **Hit Rate@K**: Top-K 안에 정답이 하나라도 있으면 hit
- **MRR@K**: 정답이 몇 번째에 등장했는지의 역수 평균


In [55]:
eval_queries = [
    {"query": "안전한 배당 ETF", "relevant": ["ACE 미국배당다우존스", "TIGER 국고채10년", "KODEX 단기채권PLUS"]},
    {"query": "미국 기술주 투자", "relevant": ["KODEX 미국나스닥100TR", "TIGER 미국필라델피아반도체"]},
    {"query": "2차전지 관련 ETF", "relevant": ["TIGER 2차전지테마"]},
    {"query": "금 투자 인플레이션 방어", "relevant": ["KODEX 골드선물(H)"]},
    {"query": "국내 대형주 인덱스", "relevant": ["KODEX 200"]},
    {"query": "부동산 배당 투자", "relevant": ["TIGER 리츠부동산인프라"]},
    {"query": "저비용 미국 ETF", "relevant": ["KODEX 미국S&P500TR", "ACE 미국배당다우존스"]},
    {"query": "채권 안정 수익", "relevant": ["TIGER 국고채10년", "KODEX 단기채권PLUS"]},
]

In [57]:
# hybrid_search, vectorstore.similarity~~, bm25_search
def hit_rate(eval_data, search_fn, k=5):
  hits = 0
  for item in eval_data:
    results = search_fn(item['query'], k=k)
    found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results] # 튜플 형태로 있다면 r[0], 튜플의 첫번째를 뽑음
    if any(rel in found for rel in item['relevant']):
      hits += 1

  return hits / len(eval_data)

In [58]:
def vec_fn(q, k=5):
  results = vectorstore.similarity_search(q, k=k)
  return [(doc.metadata['name'], 1.0) for doc in results]

def bm25_fn(q, k=5):
  results = bm25_search(q, k=k)
  return [(doc.metadata['name'], score) for doc, score in results]

def hybrid_fn(q, k=5):
  return hybrid_search(q, alpha = 0.5, k=k)

In [61]:
for fn in [vec_fn, bm25_fn, hybrid_fn]:
  hr3 = hit_rate(eval_queries, fn, k=3)
  hr5 = hit_rate(eval_queries, fn, k=5)
  print(f"name: {fn} | hr3: {hr3} | hr5 : {hr5}")

name: <function vec_fn at 0x7f0ffdf47740> | hr3: 0.25 | hr5 : 0.25
name: <function bm25_fn at 0x7f0ffdf471a0> | hr3: 0.25 | hr5 : 0.25
name: <function hybrid_fn at 0x7f0ffdf47100> | hr3: 0.25 | hr5 : 0.25


### MRR (Mean Reciprocal Rank)

정답이 검색 결과의 몇 번째에 나왔는지를 역수로 점수화한 평가 지표.

- 1등: 1, 2등: 1/2, 3등: 1/3, ...
- 여러 쿼리에 대해 Reciprocal Rank 를 평균 → MRR
- MRR@K 는 상위 K 안에서만 계산, K 밖이면 0점


In [62]:
def mrr(eval_data, search_fn, k=5): # MRR@K
  rr_sum = 0

  for item in eval_data:
    results = search_fn(item['query'], k=k)
    found = [r[0] if isinstance(r, tuple) else r.metadata.get('name', '') for r in results]

    for rank, name in enumerate(found, start=1):
        if name in item['relevant']:
            rr_sum += 1/rank
            break

  return rr_sum / len(eval_data)

In [63]:
for fn in [vec_fn, bm25_fn, hybrid_fn]:
  hr3 = hit_rate(eval_queries, fn, k=3)
  hr5 = hit_rate(eval_queries, fn, k=5)
  mrr3 = mrr(eval_queries, fn, k=3)
  mrr5 = mrr(eval_queries, fn, k=5)
  print(f"name: {fn} | hr3: {hr3} | hr5 : {hr5} | mrr 3: {mrr3} | mrr5 : {mrr5}")

name: <function vec_fn at 0x7f0ffdf47740> | hr3: 0.25 | hr5 : 0.25 | mrr 3: 0.25 | mrr5 : 0.25
name: <function bm25_fn at 0x7f0ffdf471a0> | hr3: 0.25 | hr5 : 0.25 | mrr 3: 0.25 | mrr5 : 0.25
name: <function hybrid_fn at 0x7f0ffdf47100> | hr3: 0.25 | hr5 : 0.25 | mrr 3: 0.25 | mrr5 : 0.25


## 검색 결과 보정 — Reranking

초기 검색 결과를 한 번 더 정렬/보정하여 품질 향상.

- MMR (Maximum Marginal Relevance): 유사도 + 다양성 균형
- LLM Listwise Reranking: 후보 전체를 LLM 에게 보여주고 재정렬 요청


### MMR (Maximum Marginal Relevance)

유사도와 다양성을 동시에 고려해 결과를 고르는 방식.

- `vectorstore.max_marginal_relevance_search(query, k, fetch_k)`
- 이미 선택된 문서와 **너무 유사한** 후보는 패널티
- 추천·요약처럼 "중복 없는 다양한 결과" 가 필요한 상황에 유용


In [64]:
# MMR (Maximum Marginal Relevance)
# 검색 결과를 다양하게 하는 법

In [67]:
query = "좋은 투자 상품 추천"
sim_results = vectorstore.similarity_search(query, k=5)
mmr_results = vectorstore.max_marginal_relevance_search(query, k=5, fetch_k = 12, lambda_multi=0.5)
# mmr = lambda * similarity + (1-lambda) * diversity

for doc in sim_results:
  print(f" {doc.metadata['name']} | {doc.metadata['category']}")
print("="*100)
for doc in mmr_results:
  print(f" {doc.metadata['name']} | {doc.metadata['category']}")

 TIGER 2차전지테마 | 테마
 TIGER 미국필라델피아반도체나스닥 | 해외주식
 ACE KRX금현물 | 배당
 KODEX 200 | 국내주식
 KODEX 미국나스닥100 | 해외주식
 TIGER 2차전지테마 | 테마
 KODEX 200 | 국내주식
 ACE KRX금현물 | 배당
 TIGER 미국필라델피아반도체나스닥 | 해외주식
 KODEX 미국나스닥100 | 해외주식


In [68]:
import re

### LLM Listwise Reranking

초기 후보 리스트 전체를 LLM 에게 제시하고 "쿼리와 가장 잘 맞는 순서" 로 재정렬 요청.

- `llm_rerank(query, candidates, k)` — 이름 목록만 반환받아 원본 Document 와 매칭
- 후보 수가 적을수록 비용·정확도 trade-off 유리


In [72]:
def llm_rerank(query, candidates, k=5): # 검색 결과가 candidate에 저장되어 있을 거고, reranking을 통해서 추려낼것
  name_to_doc = {d.metadata['name'] : d for d in documents}
  cand_text = ""
  for i, (name, score) in enumerate(candidates):
    doc = name_to_doc.get(name)
    if doc:
      m = doc.metadata
      cand_text += (f"{i+1}, {name} ({m['category']}) |"
      f"수수료 {m['expense_ratio']}% | 수익률 {m['return_1y']}% |"
      f"배당 {m['dividend_yield']}% \n"
      )

  prompt = f"""사용자 쿼리에 가장 적합한 ETF를 순위대로 정렬하세요.

  쿼리 : {query}
  후보 ETF : {cand_text}
  가장 적합한 {k} 개를 순위대로 번호만 응답하세요.
  예 : 3, 1, 5, 2, 4"""

  response = llm.invoke([{'role' : 'user', 'content': prompt}]).content
  numbers = [int(x) for x in re.findall(r'\d+', response)]

  reranked = []
  seen = set()
  for n in numbers:
    if 1 <= n <= len(candidates) and n not in seen:
      reranked.append(candidates[n-1])
      seen.add(n)

  for c in candidates:
    if c not in reranked:
      reranked.append(c)

  return reranked[:k]

In [73]:
query = "초보자에게 안전한 ETF"
initial = hybrid_search(query, alpha=0.5, k=8)
reranked = llm_rerank(query, initial, k=5)

In [76]:
initial

[('TIGER 2차전지테마', np.float64(1.0)),
 ('TIGER 미국필라델피아반도체나스닥', np.float64(0.8723911046981812)),
 ('KODEX 미국S&P500', np.float64(0.7557119859227244)),
 ('KODEX 200', np.float64(0.25028414137397387)),
 ('ACE KRX금현물', np.float64(0.2265592233255441)),
 ('KODEX 미국나스닥100', np.float64(0.0))]

In [74]:
reranked

[('KODEX 미국S&P500', np.float64(0.7557119859227244)),
 ('ACE KRX금현물', np.float64(0.2265592233255441)),
 ('KODEX 200', np.float64(0.25028414137397387)),
 ('KODEX 미국나스닥100', np.float64(0.0)),
 ('TIGER 2차전지테마', np.float64(1.0))]

## 콘텐츠 기반 필터링 (Content-based Filtering)

아이템의 속성 벡터만으로 유사 아이템을 추천. 사용자 기록이 없어도 동작 (콜드스타트 강점).

| 방식 | 입력 | 장단점 |
|---|---|---|
| Content-based | 아이템 속성 | 콜드스타트 OK, 속성 설계 의존 |
| Collaborative | 사용자-아이템 상호작용 | 사용자 취향 반영, 데이터 필요 |


In [77]:
# contents-based filtering
  # A: 짜장면 좋아함 => 짜장면 맛집만 추천해줌
  # B: 탕수육 => 탕수육 맛집만 추천해줌

  # ETF의 경우
  # A: 안전 추구형 => 안전한 상품만 추천
  # B: 높은 수익률 => ...

In [ ]:
# 요즘은 Collaborative FIltering (협업 기반 추천, 협업 기반 필터링)
        # 1번 제품, 2번, 3, 4, 5, 6, 7 ...
# A사람    O         O   O
# B                         O  O
# C        O         O   ?
  # 그럼 유저들끼리의 유사도를 구해서..
  # C는 A와 유사하다고 판단함, C한테 3번을 추천해줌

# 데이터를 다뤄봤으면 좀 말이 안될 수 있음
  # 1억개가 넘는 상품이 있다고 가정..
# A: 1번, 5000번 구매
# B: 2번, 8000번 구매
# C: 1번만 구매
# 모든 데이터에 적용하기가 쉽지 않음, 행렬 (1억열, 1천만 행), 고차원 데이터
  # 대부분의 값이 비어있음..
  # 이런걸 sparse matrix 라고 함

# 행렬 -> 차원 분해/ 축소 등의 방법으로 처리

### ETF 속성 벡터화 & 코사인 유사도

- `risk_map` 등으로 범주형 속성을 수치로 매핑
- `etf_to_vector(etf)` — 보수율·수익률·배당률·변동성·위험등급 등을 하나의 벡터로 변환
- `cosine_sim(a, b) = a·b / (||a|| · ||b||)` — 방향 기반 유사도


In [78]:
etf_data = [
    {"name": "KODEX 200",             "category": "국내주식", "expense_ratio": 0.15, "return_1y":  8.5, "risk_level": "중간", "dividend_yield": 1.8, "volatility": 15.2},
    {"name": "KODEX 미국S&P500TR",    "category": "해외주식", "expense_ratio": 0.05, "return_1y": 25.3, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 18.5},
    {"name": "ACE 미국배당다우존스",   "category": "배당",     "expense_ratio": 0.01, "return_1y": 12.1, "risk_level": "낮음", "dividend_yield": 3.5, "volatility": 10.3},
    {"name": "TIGER 2차전지테마",     "category": "테마",     "expense_ratio": 0.45, "return_1y":-15.2, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 35.7},
    {"name": "TIGER 국고채10년",      "category": "채권",     "expense_ratio": 0.07, "return_1y":  4.2, "risk_level": "낮음", "dividend_yield": 2.8, "volatility":  5.1},
    {"name": "KODEX 골드선물(H)",     "category": "원자재",   "expense_ratio": 0.68, "return_1y": 18.7, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 20.4},
    {"name": "KODEX 미국나스닥100TR", "category": "해외주식", "expense_ratio": 0.05, "return_1y": 32.1, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 25.3},
    {"name": "TIGER 미국반도체",      "category": "섹터",     "expense_ratio": 0.49, "return_1y": 45.0, "risk_level": "높음", "dividend_yield": 0.0, "volatility": 38.2},
    {"name": "KODEX 단기채권PLUS",    "category": "채권",     "expense_ratio": 0.03, "return_1y":  3.8, "risk_level": "낮음", "dividend_yield": 3.2, "volatility":  1.5},
    {"name": "TIGER 리츠부동산",      "category": "리츠",     "expense_ratio": 0.29, "return_1y":  6.5, "risk_level": "중간", "dividend_yield": 4.8, "volatility": 12.8},
    {"name": "KODEX 200TR",           "category": "국내주식", "expense_ratio": 0.12, "return_1y":  9.1, "risk_level": "중간", "dividend_yield": 0.0, "volatility": 14.9},
    {"name": "TIGER 고배당저변동",    "category": "배당",     "expense_ratio": 0.30, "return_1y":  7.2, "risk_level": "낮음", "dividend_yield": 5.2, "volatility":  8.7},
]

In [79]:
etf_df = pd.DataFrame(etf_data)
etf_df.head()

# 모든 속성을 벡터로 묶은 후에, 선호도에 대해서 벡터가 유사한 것들을 추천해줌
  # 벡터로 만들때 문제가 되는것.. -> '중간', '낮음', '높음' 등등

,name,category,expense_ratio,return_1y,risk_level,dividend_yield,volatility
0,KODEX 200,국내주식,0.15,8.5,중간,1.8,15.2
1,KODEX 미국S&P500TR,해외주식,0.05,25.3,중간,0.0,18.5
2,ACE 미국배당다우존스,배당,0.01,12.1,낮음,3.5,10.3
3,TIGER 2차전지테마,테마,0.45,-15.2,높음,0.0,35.7
4,TIGER 국고채10년,채권,0.07,4.2,낮음,2.8,5.1


In [81]:
# 우선 contents-based filtering

risk_map = {'낮음':1, '중간': 2, '높음':3}

def etf_to_vector(etf): # 벡터화를 하긴 하는데, 속성간의 크기가 서로 다름
  return np.array([
      # 임시로 정규화 처리
      risk_map[etf['risk_level']] / 3,
      (etf['return_1y'] + 30)/80,
      etf['dividend_yield'] / 6,
      etf['expense_ratio'],
      etf['volatility'] / 40 # (정규화 안한다면)얘가 제일 값이 큰 속성이므로 여기로 치우칠 가능성이 높음
  ])

In [85]:
from numpy.linalg import norm
def cosine_sim(a, b):
  return float(np.dot(a,b) / (norm(a) * norm(b) + 1e-8))

In [87]:
item_vectors = np.array([etf_to_vector(e) for e in etf_data])
item_names = [e['name'] for e in etf_data]

In [89]:
item_vectors.shape

(12, 5)

### 아이템 간 유사도 행렬 — `build_item_similarity`

- 전 아이템 쌍에 대해 코사인 유사도 사전 계산 → `item_sim[i, j]` 조회용 행렬
- 추천 시 인덱싱 한 번으로 Top-K 산출


In [90]:
def build_item_similarity(vectors):
  n = len(vectors)
  sim = np.zeros((n,n))
  for i in range(n):
    for j in range(n):
      sim[i, j] = cosine_sim(vectors[i], vectors[j])

  return sim

In [91]:
item_sim = build_item_similarity(item_vectors)

In [92]:
item_sim

array([[0.99999999, 0.93176159, 0.87081043, 0.8439353 , 0.88918454,
        0.86267631, 0.94177404, 0.92426011, 0.82408998, 0.91011453,
        0.94997665, 0.78277067],
       [0.93176159, 0.99999999, 0.74171806, 0.80825004, 0.74489487,
        0.85788705, 0.99156833, 0.95861508, 0.65807283, 0.71069736,
        0.98424562, 0.56030593],
       [0.87081043, 0.74171806, 0.99999999, 0.52313937, 0.98634932,
        0.61579404, 0.71707589, 0.69424573, 0.9670778 , 0.93551631,
        0.70712417, 0.9363389 ],
       [0.8439353 , 0.80825004, 0.52313937, 1.        , 0.5358344 ,
        0.87124698, 0.8570988 , 0.90483416, 0.4235036 , 0.67303315,
        0.87685179, 0.47605488],
       [0.88918454, 0.74489487, 0.98634932, 0.5358344 , 0.99999998,
        0.65748806, 0.72496447, 0.70048449, 0.98798759, 0.9594121 ,
        0.72957456, 0.94811582],
       [0.86267631, 0.85788705, 0.61579404, 0.87124698, 0.65748806,
        0.99999999, 0.85189897, 0.95314591, 0.55126458, 0.71788042,
        0.89372903,

### 유사 ETF 추천 & 다양성 보장 변형

- `cbf_similar_items(target_name, top_k)` — 타겟과 가장 유사한 ETF 상위 K 개
- `cbf_diverse(target_name, top_k)` — 카테고리 중복을 피해 다양한 ETF 추천


In [96]:
# item_sim
def cbf_similar_items(target_name, top_k=5):
  idx = item_names.index(target_name)
  scores = [(item_names[j], item_sim[idx, j ]) for j in range(len(item_names)) if j != idx]
  return sorted(scores, key = lambda x:x[1], reverse=True)[:top_k]

In [98]:
for name, score in cbf_similar_items('KODEX 200', top_k=5):
  print(f" {score} | {name}")

 0.9499766521932036 | KODEX 200TR
 0.9417740369716172 | KODEX 미국나스닥100TR
 0.9317615934699991 | KODEX 미국S&P500TR
 0.9242601096073657 | TIGER 미국반도체
 0.9101145298708155 | TIGER 리츠부동산


In [99]:
a = [1,2,3,5,4]
a.sort()
a

[1, 2, 3, 4, 5]

In [108]:
# 최대한 다양한 카테고리에서 뽑고 싶다면..?
def cbf_diverse(target_name, top_k=5): # top_k개 만큼 뽑으나, 겹치지 않게 뽑아야함
  idx = item_names.index(target_name)
  scores = [(j, item_sim[idx, j ]) for j in range(len(item_names)) if j != idx]
  scores.sort(key = lambda x: x[1], reverse = True)

  result, seen_cats = [], set()
  for j, score in scores:
    cat = etf_data[j]['category']
    if cat in seen_cats:
      continue
    result.append((item_names[j], round(score,3), cat))
    seen_cats.add(cat)
    if len(result) >= top_k:
      break

  return result

In [109]:
cbf_diverse('KODEX 200', top_k=5)

[('KODEX 200TR', np.float64(0.95), '국내주식'),
 ('KODEX 미국나스닥100TR', np.float64(0.942), '해외주식'),
 ('TIGER 미국반도체', np.float64(0.924), '섹터'),
 ('TIGER 리츠부동산', np.float64(0.91), '리츠'),
 ('TIGER 국고채10년', np.float64(0.889), '채권')]